In [ ]:
# These lines import the Numpy and Datascience modules.
import numpy as np
from datascience import *

# These lines do some fancy plotting magic
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')
import warnings
warnings.simplefilter('ignore', FutureWarning)
from matplotlib import patches
from ipywidgets import interact, interactive, fixed
import ipywidgets as widgets


## So from prior experience and the internet, Skittles are supposed to be uniformly distributed.  That is, the population of all Skittles should have the same proportion of each color -- 20%.  However, from our box that we counted earlier this week, this appears to be no longer the case.  To that end, I want to perform a TVD hypothesis test to see if what we observed in our class sample is due to chance alone or there really may be a different mix these days.

### H0: The Skittles colors are uniformly distributed, with 20% of each color
### Ha: Skittles colors are not uniformly distributed

In [ ]:
skittles = Table().with_columns(
    'Color', make_array('Red', 'Yellow', 'Purple', 'Orange', 'Green'),
    'Proportion', make_array(0.2, 0.2, 0.2, 0.2, 0.2) 
)

skittles

## Let's get our data and then create a column to add to this table

In [ ]:
link = 'https://docs.google.com/spreadsheets/d/12EMZTyOv-k1r6mg3BUm9uuKKAfPtb2Cxyv7z3_MRtq8/export?format=csv'


our_skittles = Table().read_table(link).drop('Timestamp')
our_skittles

In [ ]:
red_total = sum(our_skittles.column("Red"))
yellow_total = sum(our_skittles.column("Yellow"))
purple_total = sum(our_skittles.column("Purple"))
orange_total = sum(our_skittles.column("Orange"))
green_total = sum(our_skittles.column("Green"))
grand_total = red_total + yellow_total + purple_total + orange_total + green_total
print(grand_total)

prop_array = make_array(red_total/grand_total, yellow_total/grand_total, purple_total/grand_total, orange_total/grand_total, green_total/grand_total)

#add our proportions to the skittles table
skittles = skittles.with_column("Our Proportions", prop_array)
skittles

## Let's look at a bar chart of the data

In [ ]:
skittles.barh("Color")

## In the cell below, sample at random 1947 times (the grand total) from the distribution of theoretical proportions, and display the distribution of the random sample along with the theoretical distribution

In [ ]:
#get the theoretical proportions
theo_prop = skittles.column('Proportion')
sample_distribution = sample_proportions(grand_total, theo_prop)
skittles_and_sample = skittles.with_column('Random Sample', sample_distribution)
skittles_and_sample

In [ ]:
skittles_and_sample.barh("Color")

### Now, let's calculate the Total Variation Distance.  Write a function calculate_tvd that takes in the observed distribution (obs_dist) and expected distribution under the null hypothesis (null_dist) and calculates the total variation distance. Use this function to set observed_tvd to be equal to the observed test statistic. 

In [ ]:
null_distribution = skittles.column("Proportion")
print(null_distribution)


def calculate_tvd(obs_dist, null_dist):
    abs_distances = abs(obs_dist - null_dist)
    tvd = sum(abs_distances) / 2
    return tvd
    
observed_tvd = calculate_tvd(skittles.column("Our Proportions"), null_distribution)
print(observed_tvd)

### Create an array called simulated_tvds that contains 1,000 simulated values under the null hypothesis. Recall that the original sample consisted of 1947 Skittles

In [ ]:
simulated_tvds = make_array()

for i in np.arange(1000):
    a_simulation = sample_proportions(grand_total, null_distribution)
    a_simulated_tvd = calculate_tvd(a_simulation, null_distribution)
    simulated_tvds = np.append(simulated_tvds, a_simulated_tvd)



### Plot a histogram of the simulated test statistics, as well as a red dot representing the observed value of the test statistic.

In [ ]:
Table().with_column("Simulated TVDs", simulated_tvds).hist()
plt.scatter(observed_tvd, 0.5, color='red', s=70, zorder=2);
plt.show();

### Calculate the p-value

In [ ]:
p_value_tvd = sum(simulated_tvds > observed_tvd)/ 1000
p_value_tvd

### Decision

### Conclusion